# **Step 1 – Entendimiento del dominio y del objetivo**
**Proyecto TA – Análisis de Datos (1INF03, 2026-2) – Equipo E10**

> *KDD, paso 1 (Fayyad et al.):* «Developing an understanding of the application domain and the relevant prior knowledge, and identifying the goal of the KDD process from the customer's viewpoint.»

Este notebook resume el caso, el conocimiento previo necesario y los objetivos del análisis. Es mayormente texto; el código solo inventaría las fuentes de datos.

## 1. El caso

El **Puericultorio Pérez Araníbar** acoge a niñas, niños y adolescentes (NNA) en situación de desprotección familiar, organizados en hogares (Infantes, Miguel Echenique, Tomás Valle, Matogroso Mujeres y Matogroso Hombres).

Cada mes la institución llena una **ficha nominal** por NNA (datos personales, situación legal, plan de trabajo, educación, salud, soporte psicológico). Uno de sus campos es **`Nivel Nutricional`**, que se escribe **a mano** con texto libre (p. ej. `Saludable`, `Sobrepeso y riesgo de talla baja`, `En proceso de evaluación`).

Aparte, la nutricionista realiza **evaluaciones antropométricas** (peso, talla, IMC) y clasifica cada medida con dos referencias: **OMS** (patrones internacionales de crecimiento) e **INS** (tabla de diagnóstico rápido del Instituto Nacional de Salud, Perú).

**Problema de negocio:** no se sabe qué tan bien el `Nivel Nutricional` de la ficha refleja las mediciones. Si el registro está desactualizado o es inconsistente, la institución puede dejar de atender a tiempo a NNA con exceso de peso o retraso de talla.

## 2. Conocimiento previo del dominio

**Indicadores antropométricos** (según la descripción de la institución):

| Indicador | Qué relaciona | Para qué sirve |
|---|---|---|
| **P/E** – peso para la edad | peso vs. edad | peso adecuado para la edad |
| **T/E** – talla para la edad | talla vs. edad | crecimiento lineal (retraso de talla) |
| **P/T** – peso para la talla | peso vs. talla | estado nutricional combinando ambas medidas |
| **IMC** – índice de masa corporal | peso / talla² (kg/m²) | exceso o déficit de peso para la edad |

**Indicadores según la edad:** en **menores de 5 años** se usan P/E, T/E y P/T; en **mayores de 5 años y adolescentes**, T/E e IMC. Por eso es esperable que P/E y P/T estén vacíos en los mayores (vacío estructural, no error).

**Frecuencia:** los infantes se evalúan cada 1–2 meses y los adolescentes cada ≈3 meses; no todos los hogares se miden todos los meses.

**Implicancia clave:** el estado nutricional no depende de la medida cruda sino de **cuánto se aleja de lo esperado para la edad y el sexo** (así funcionan las curvas OMS). Un niño de 10 años pesa más que uno de 3 sin que eso indique exceso.

## 3. Objetivos del proceso KDD (vista de la institución)

1. **Principal – clasificación:** estimar, con peso, talla, IMC, edad y sexo, si un NNA tiene **alteración nutricional estricta** según la ficha (exceso de peso o retraso de talla), y medir cuánto aportan las medidas. La etiqueta amplia (que incluye *riesgos*) se reporta como análisis de sensibilidad.
2. **Auditoría:** identificar NNA y meses en que la ficha y la clasificación OMS de las mediciones **discrepan**, como lista de revisión para la nutricionista.
3. **Complementario (entrega final):** perfiles de NNA (clustering), solo exploratorio en el avance.

**Hipótesis inicial descartada:** predecir la alteración usando solo las variables administrativas de la ficha no funcionó (ver `Anexo_Step4_solo_ficha` y `Anexo_Step6_solo_ficha`). Ese resultado negativo motivó el enfoque actual con mediciones.

**Criterio de éxito:** un modelo que supere claramente a una línea base (AUC-ROC en validación agrupada por niño, distinguible del azar) y una lista de discrepancias accionable.

## 4. Inventario de las fuentes de datos
Se listan los archivos de `datos/0_crudos/` (copia exacta de los originales entregados por la institución) con su tamaño y contenido básico.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import openpyxl
RAW = Path("../datos/0_crudos")
RES = Path("../datos/4_resultados")

Se recorre cada carpeta de crudos y se cuenta, para cada archivo, cuántas filas de NNA contiene (filas con un ID `PPA_FN_…`).

In [ ]:
filas = []
ficha = pd.read_csv(RAW/"ficha_nominal/FichaNominal_consolidado_IDs_v2.txt", sep="|", dtype=str, keep_default_na=False)
filas.append({"bloque": "1 – Ficha nominal", "archivo": "FichaNominal_consolidado_IDs_v2.txt", "hojas": 1,
              "filas_NNA": len(ficha), "periodos": f"{ficha['periodo'].min()} a {ficha['periodo'].max()} ({ficha['periodo'].nunique()})"})
for f in sorted((RAW/"nutricion_mensual").glob("*.xlsx")):
    wb = openpyxl.load_workbook(f, read_only=True, data_only=True)
    n = sum(1 for ws in wb.worksheets for r in ws.iter_rows(max_col=3, values_only=True)
            if isinstance(r[1], str) and r[1].strip().startswith("PPA_FN_"))
    filas.append({"bloque": "2 – Nutrición 2026", "archivo": f.name, "hojas": len(wb.worksheets), "filas_NNA": n,
                  "periodos": f.name[14:18] + "_" + f.name[18:20]})
for f in sorted((RAW/"referencias").glob("*.pdf")):
    filas.append({"bloque": "Referencia", "archivo": f.name, "hojas": None, "filas_NNA": None, "periodos": None})
inventario = pd.DataFrame(filas)
inventario

**Interpretación.** El proyecto usa dos bloques de datos:
- **Bloque 1 – Ficha nominal** (un archivo, 2 667 filas, 19 periodos mensuales de 2025_01 a 2026_07): contiene la etiqueta `Nivel Nutricional`.
- **Bloque 2 – Nutrición 2026** (6 Excel mensuales: enero, febrero, marzo, abril, junio y agosto; no hay mayo ni julio): contiene las mediciones. Cada Excel tiene una hoja por hogar con columnas que **varían** entre hojas y meses; por eso el Step 2a los consolida con un mapeo explícito.
- **Referencias:** tablas del CENAN/INS y curvas OMS en PDF (material de consulta, no se procesan).



## 5. Verificación y cifras de este notebook

In [ ]:
n_nutri = int((inventario["bloque"] == "2 – Nutrición 2026").sum())
assert len(ficha) == 2667 and ficha["periodo"].nunique() == 19
assert n_nutri == 6
cifras = {"filas_ficha": len(ficha), "periodos_ficha": int(ficha["periodo"].nunique()), "archivos_nutricion": n_nutri}
json.dump(cifras, open(RES/"cifras_step1.json", "w"), indent=1); cifras